[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresOrdonez369/deepLearningWeek1/blob/main/proyectos/NB01_viernes_tu_primera_neurona.ipynb)

# NB01 · Tu primera neurona
**Neural Networks: Deep Learning · UIDE · Semana 1 · Viernes** — Docente: Andrés Ordóñez-Bolaños

En este Colab vas a **construir una neurona con tus propias manos**: verás que tu nota final del curso ya es una neurona, la pondrás a decidir créditos, la verás equivocarse y aprender, y al final bajarás una "montaña" con descenso de gradiente. Todo se explica paso a paso; no necesitas instalar nada.

### Cómo usar este Colab (2 minutos)
1. **Archivo → Guardar una copia en Drive.** Así tus cambios quedan guardados en tu cuenta.
2. Ejecuta la celda **Configuración** (la primera de código). Después avanza en orden: clic en la celda y `Shift + Enter`.
3. Si algo raro pasa: **Entorno de ejecución → Ejecutar todo** vuelve a correr todo desde el principio.
4. Cada sección tiene el mismo orden: **Qué vas a hacer → la idea → ejemplo resuelto (ejecútalo y léelo) → Qué deberías ver → Tu turno**.
5. En **Tu turno** reemplaza los `None` y ejecuta la celda de revisión. Te responde:
   - ✅ correcto · ❌ todavía no (con una pista) · ⏳ todavía no lo completas.
6. **No tiene nota.** Equivocarse aquí es parte del plan.

### Mapa del Colab
| Parte | Qué aprendes | Tu turno | Lámina |
|---|---|---|---|
| 0 | Python exprés: variables, listas, funciones, NumPy y gráficas (15 min, **si ya sabes Python, sáltala**) | 0A, 0B | — |
| 1 | Tu nota final es una neurona · el comité de crédito | 1A, T1 | 12–14 |
| 2 | La activación: de puntaje a decisión | T2 | 16–17 |
| 3 | El perceptrón corrige un error | T3 | 21–22 |
| 4 | Entrenar un perceptrón completo | T4 | 23 |
| 5 | El límite: XOR, la luz de la escalera | T5 | 24 |
| 6 | La pérdida: qué tan mal vamos | T6 | 28–29 |
| 7 | Descenso de gradiente: bajar el Cotopaxi | T7 | 30–32 |
| 8 | El tamaño del paso η | T8 | 33 |
| 🏁 | Resumen y **Tareas finales** (incluye el Mini-reto 1, opcional) | 3 tareas | 35 |

En clase: secciones **1 a 5** en la práctica 1 (lámina 26) y **6 a 8** en la práctica 2 (lámina 35).

In [ ]:
# @title Configuración y autocorrección · ejecuta esta celda primero (no necesitas abrirla)
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2026)  # fixed seed -> everyone gets the same results
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

# The same colors as the slides: amber = yes / +1, azure = no / -1, mint = learned, coral = error
AMBAR, AZUL, MENTA, CORAL, TINTA = "#E39A12", "#2B8FD6", "#12A37F", "#E0533D", "#1F2A3A"

# ---------- self-check helpers (formative feedback, no grades) ----------
def _ok(msg):   print("✅ " + msg)
def _no(msg):   print("❌ " + msg)
def _todo():    print("⏳ Aún no completas esta parte: reemplaza los None y vuelve a ejecutar la celda.")
def _num(v):
    try:
        return float(np.asarray(v, dtype=float).reshape(-1)[0]) if np.size(v) == 1 else np.asarray(v, dtype=float)
    except Exception:
        return None
def _near(a, b, tol=1e-2):
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol))
    except Exception:
        return False
_sig = lambda z: 1 / (1 + np.exp(-z))
_PESOS = np.array([0.8 / 3, 0.8 / 3, 0.8 / 3, 0.2])

def revisar_0a(nota_beto):
    if nota_beto is None:
        return _todo()
    if not _near(nota_beto, 62.0):
        return _no(f"La nota de Beto debería ser 62.0 y obtuviste {_num(nota_beto)}. Usa nota_final([60, 55, 65, 70], pesos).")
    _ok("Beto: 62.0. Las tres semanas aportan 48 puntos y el entrega final 14.")

def revisar_0b(notas_grupo):
    if notas_grupo is None:
        return _todo()
    if not _near(notas_grupo, [76.0, 62.0, 86.0]):
        return _no(f"Esperaba [76. 62. 86.] y obtuve {_num(notas_grupo)}. Pista: grupo @ w (matriz @ vector).")
    _ok("[76. 62. 86.]: tres notas finales con una sola operación. Así calcula una capa entera de neuronas.")

def revisar_1a(final_necesario):
    if final_necesario is None:
        return _todo()
    if not _near(final_necesario, 30.0, 0.05):
        return _no(f"Obtuviste {_num(final_necesario)}. Pista: z = 0 justo en el límite → 64 + 0.2·F − 70 = 0. Despeja F.")
    _ok("Con 30 en el entrega final Ana llega justo a 70: z = 0, la frontera entre aprobar y no aprobar.")

def revisar_t1(neurona, z_b):
    if neurona is None or z_b is None:
        return _todo()
    try:
        out = neurona(np.array([0.8, 0.5, -1.0]), np.array([1.2, 0.7, -2.0]), -0.5)
    except Exception as e:
        return _no(f"Tu función produjo un error: {e}")
    if out is None:
        return _no("Tu función devuelve None. Pista: return w @ x + b")
    if not _near(out, 2.81):
        return _no(f"Con el socio A debería dar 2.81 y da {_num(out)}. ¿Sumaste el sesgo b?")
    rng = np.random.default_rng(1)
    for _ in range(5):
        x, w, b = rng.normal(size=4), rng.normal(size=4), rng.normal()
        if not _near(neurona(x, w, b), w @ x + b, 1e-6):
            return _no("Funciona con 3 datos, pero no con otros vectores. ¿Escribiste los números a mano?")
    if not _near(z_b, -3.61):
        return _no(f"La función está bien ✔, pero z_b debería ser -3.61 y es {_num(z_b)}.")
    _ok("¡Perfecto! Tu neurona funciona y el socio B obtiene z = -3.61: el comité lo rechaza.")

def revisar_t2(p_b, decision_b):
    if p_b is None or decision_b is None:
        return _todo()
    if not _near(p_b, _sig(-3.61), 1e-3):
        return _no(f"p_b debería ser sigmoid(z_b) ≈ 0.026 y es {_num(p_b)}.")
    if str(decision_b).strip().lower() not in ("rechaza", "rechazar", "no"):
        return _no("Con 2.6 % de probabilidad de pago y umbral 0.5, la decisión es 'rechaza'.")
    _ok("Correcto: 2.6 % de probabilidad de pago → el comité rechaza.")

def revisar_t3(w_nuevo, b_nuevo, todavia_se_equivoca):
    if w_nuevo is None or b_nuevo is None or todavia_se_equivoca is None:
        return _todo()
    if not _near(w_nuevo, [0.2, 0.0]):
        return _no(f"w_nuevo debería ser w + η·y·x = (0.2, 0.0) y es {_num(w_nuevo)}. Ojo: aquí y = -1.")
    if not _near(b_nuevo, 0.0):
        return _no(f"b_nuevo debería ser b + η·y = 0.0 y es {_num(b_nuevo)}.")
    if todavia_se_equivoca is not True:
        return _no("Calcula z con los pesos nuevos: 0.2·1 + 0·0.5 + 0 = 0.2 ≥ 0, así que todavía aprueba (se equivoca).")
    _ok("¡Exacto! Una corrección acercó la frontera, pero no bastó: el perceptrón necesita varias vueltas.")

def revisar_t4(w_and, b_and, X_and, y_and):
    if w_and is None or b_and is None or X_and is None or y_and is None:
        return _todo()
    X_and, y_and = np.asarray(X_and, float), np.asarray(y_and)
    if X_and.shape != (4, 2) or sorted(map(tuple, X_and)) != [(0, 0), (0, 1), (1, 0), (1, 1)]:
        return _no("X_and debe tener las 4 combinaciones de dos interruptores: (0,0), (0,1), (1,0), (1,1).")
    truth = np.array([1 if (a == 1 and b == 1) else -1 for a, b in X_and])
    if not np.array_equal(y_and, truth):
        return _no("y_and: AND solo es +1 cuando ambos valen 1; en los otros tres casos es -1.")
    pred = np.where(X_and @ np.asarray(w_and, float) + b_and >= 0, 1, -1)
    if not np.array_equal(pred, y_and):
        return _no("Los pesos aprendidos todavía no clasifican bien los 4 casos. ¿Entrenaste con train_perceptron?")
    _ok(f"¡Aprendió AND con 0 errores! w = {np.round(w_and, 2)}, b = {b_and}. AND sí se puede separar con una línea.")

def revisar_t5(respuesta_t5):
    if respuesta_t5 is None:
        return _todo()
    if str(respuesta_t5).strip().lower() == "b":
        return _ok("Correcto: no existe UNA línea recta que deje (0,1) y (1,0) de un lado y (0,0) y (1,1) del otro.")
    hints = {"a": "Más épocas no ayudan: en la gráfica los errores se repiten igual para siempre.",
             "c": "Con cualquier η pasa lo mismo: el problema no es el tamaño del paso.",
             "d": "Los 4 datos de XOR no tienen ruido: son exactos."}
    _no(hints.get(str(respuesta_t5).strip().lower(), "Elige una letra: 'a', 'b', 'c' o 'd'."))

def revisar_t6(mse, bce):
    if mse is None or bce is None:
        return _todo()
    try:
        m = mse(np.array([120, 95, 150, 80]), np.array([110, 100, 170, 82]))
        b1, b0 = bce(1, 0.9), bce(0, 0.9)
    except Exception as e:
        return _no(f"Tus funciones produjeron un error: {e}")
    if m is None and b1 is None:
        return _todo()
    if m is None or b1 is None or b0 is None:
        return _no("Una de las dos funciones todavía devuelve None: completa su return.")
    if not _near(m, 132.25):
        return _no(f"mse con las casas del ejemplo debería dar 132.25 y da {_num(m)}. ¿Elevaste al cuadrado antes de promediar?")
    if not _near(b1, -np.log(0.9), 1e-3):
        return _no(f"bce(1, 0.9) debería ser -log(0.9) ≈ 0.105 y da {_num(b1)}.")
    if not _near(b0, -np.log(0.1), 1e-3):
        return _no(f"bce(0, 0.9) debería ser -log(1 - 0.9) ≈ 2.303 y da {_num(b0)}. Usa la fórmula completa con y y (1 - y).")
    _ok("¡Bien! MSE = 132.25; BCE: decir 90 % cuando sí pagó cuesta 0.11, y cuando no pagó cuesta 2.30.")

def revisar_t7(minimo, w_final):
    if minimo is None or w_final is None:
        return _todo()
    if not _near(minimo, -2):
        return _no("(w + 2)² vale 0 cuando w + 2 = 0. ¿En qué w está el fondo?")
    if not _near(w_final, -2, 0.01):
        return _no(f"Después de 50 pasos w debería estar muy cerca de -2 y está en {_num(w_final)}. Revisa la pendiente: 2·(w + 2).")
    _ok(f"¡Llegaste al fondo! w = {float(w_final):.4f} ≈ -2.")

def revisar_t8(converge):
    if converge is None or any(v is None for v in converge.values()):
        return _todo()
    truth = {0.05: True, 0.5: True, 0.9: True, 1.1: False}
    wrong = [eta for eta in truth if converge.get(eta) != truth[eta]]
    if wrong:
        return _no(f"Revisa η = {wrong}. Pista: usa la función descenso(eta) y mira si |w - 3| se achica o crece.")
    _ok("Correcto: con η < 1 se acerca (0.05 lento, 0.9 rebotando); con η = 1.1 se aleja cada vez más.")

# Final task 2: six students (S1, S2, S3, Final)
GRUPO6 = np.array([[85, 90, 80, 75], [70, 60, 65, 80], [55, 75, 70, 90],
                   [90, 95, 85, 40], [65, 60, 55, 70], [72, 68, 74, 66]])
def revisar_tf2(notas6, aprobados):
    if notas6 is None or aprobados is None:
        return _todo()
    truth = GRUPO6 @ _PESOS
    if not _near(notas6, truth, 0.05):
        return _no(f"Las notas deberían ser {np.round(truth, 1)}. Pista: GRUPO6 @ w_nota.")
    k = int((truth - 70 >= 0).sum())
    if int(aprobados) != k:
        return _no(f"Revisa el conteo: aprueba quien tiene z = nota − 70 ≥ 0. Pista: (notas6 - 70 >= 0).sum()")
    _ok(f"{k} de 6 aprueban. Notas: {np.round(truth, 1)}. Una matriz @ un vector = la neurona aplicada a todo el grupo.")

def revisar_tf3(descenso, mi_eta):
    if mi_eta is None:
        return _todo()
    w10 = descenso(float(mi_eta), pasos=10)
    if abs(w10 - 3) < 0.01:
        return _ok(f"Con η = {mi_eta} llegas a w = {w10:.4f} en 10 pasos. Cualquier η entre ~0.22 y ~0.78 lo logra; 0.5 llega en UN paso.")
    if abs(w10 - 3) > 3:
        return _no(f"Con η = {mi_eta} te alejas (w = {w10:.2f}): el paso es demasiado largo, como abrir de golpe el agua caliente.")
    _no(f"Con η = {mi_eta} quedas en w = {w10:.4f}: todavía lejos de 3. Prueba un paso más largo (pero menor que 1).")

# Mini-reto 1 data: six members (scaled income, scaled previous delays); +1 = paid, -1 = did not pay
SOCIOS = np.array([[0.9, -0.8], [0.4, -0.2], [0.8, 0.3], [-0.5, 0.6], [0.1, 0.5], [-0.3, -0.1]])
PAGO = np.array([1, 1, 1, -1, -1, -1])
NOMBRES = ["Ana", "Byron", "Carla", "Diego", "Elena", "Fabián"]

def dibujar_comite(w, b):
    w = np.asarray(w, float)
    plt.figure(figsize=(5.5, 5))
    for (x1, x2), y, n in zip(SOCIOS, PAGO, NOMBRES):
        plt.scatter(x1, x2, s=170, c=AMBAR if y == 1 else AZUL, edgecolor=TINTA, zorder=3)
        plt.annotate(n, (x1, x2), textcoords="offset points", xytext=(9, 7))
    xs = np.linspace(-1, 1, 50)
    if abs(w[1]) > 1e-9:
        plt.plot(xs, -(w[0] * xs + b) / w[1], color=TINTA, lw=2.5, label="tu frontera (z = 0)")
    plt.xlim(-1, 1); plt.ylim(-1, 1); plt.xlabel("ingreso (escalado)"); plt.ylabel("atrasos previos (escalado)")
    plt.title("Ámbar = pagó · Azul = no pagó"); plt.legend(loc="lower left"); plt.show()

def revisar_mini_reto_1(w, b):
    if w is None or b is None:
        return _todo()
    pred = np.where(SOCIOS @ np.asarray(w, float) + b >= 0, 1, -1)
    hits = int((pred == PAGO).sum())
    dibujar_comite(w, b)
    for n, p, y in zip(NOMBRES, pred, PAGO):
        print(f"  {n:7s} → {'aprueba' if p == 1 else 'rechaza'}  {'✔' if p == y else '✘ (se equivocó)'}")
    if hits == 6:
        _ok("¡6 de 6! Tu comité decide bien con estos socios. ¿Qué pesos eligieron otros grupos?")
    else:
        _no(f"{hits} de 6 aciertos. Gira la línea con los pesos y muévela con el sesgo.")

print("Listo ✔  NumPy", np.__version__, "· autocorrección cargada")

---
## 0 · Python exprés (15 min · si ya sabes Python, sáltala o hazla antes de clase)
**Qué vas a hacer:** repasar lo mínimo de Python que usa este curso con un ejemplo que conoces bien: **tu nota final**.

Según el sílabo, tu nota del curso se calcula así (cada nota sobre 100):

$$\text{Nota final} = 0.2667\cdot S_1 + 0.2667\cdot S_2 + 0.2667\cdot S_3 + 0.20\cdot \text{Final}$$

**Léelo así:** cada semana ($S_1, S_2, S_3$) pesa 26.67 % y el entrega final pesa 20 %; multiplicas cada nota por su peso y sumas todo. Guarda esta cuenta: en la sección 1 descubrirás que **es una neurona**.

### 0.1 · Variables y `print`
Una **variable** es un nombre que guarda un valor. `print` muestra cosas en pantalla; con una **f-string** (`f"..."`) metes variables dentro del texto entre llaves `{}`.

In [ ]:
# A variable is a name that stores a value
nombre = "Ana"
s1 = 80        # week 1 (out of 100)
s2 = 70        # week 2
s3 = 90        # week 3
final = 60     # final project

print("Estudiante:", nombre)
print(f"Notas de {nombre}: S1 = {s1}, S2 = {s2}, S3 = {s3}, Final = {final}")

promedio_simple = (s1 + s2 + s3 + final) / 4
print(f"Promedio simple (todas pesan igual): {promedio_simple:.1f}")   # :.1f -> one decimal

**Qué deberías ver:** `Promedio simple (todas pesan igual): 75.0`. Pero en el curso las notas **no** pesan igual: para eso necesitamos pesos.

### 0.2 · Listas y el ciclo `for`
Una **lista** guarda varios valores en orden: `[80, 70, 90, 60]`. Un ciclo `for` recorre una lista elemento por elemento; con `zip` recorres varias listas a la vez.

In [ ]:
notas  = [80, 70, 90, 60]                       # a list: Ana's four grades, in order
pesos  = [0.8 / 3, 0.8 / 3, 0.8 / 3, 0.20]      # syllabus weights: 26.67 % each week, 20 % the final project
partes = ["S1", "S2", "S3", "Final"]

print("Los pesos suman:", round(sum(pesos), 6))   # they must add up to 1 (100 %)

nota = 0
for parte, n, p in zip(partes, notas, pesos):    # walk the three lists together
    aporte = n * p                               # what this grade contributes
    nota = nota + aporte
    print(f"{parte:5s}: {n:3d} × {p:.4f} = {aporte:5.2f}")
print(f"Nota final ponderada de Ana: {nota:.1f}")

**Qué deberías ver:** cada semana aporta 21.33, 18.67 y 24.00; el proyecto 12.00; **nota final = 76.0**.

### 0.3 · Funciones
Una **función** empaqueta una cuenta para reutilizarla: recibe datos (parámetros) y devuelve un resultado con `return`.

In [ ]:
def nota_final(notas, pesos):
    # Weighted grade: each grade times its weight, all added up.
    total = 0
    for n, p in zip(notas, pesos):
        total = total + n * p
    return total

print(f"Ana: {nota_final([80, 70, 90, 60], pesos):.1f}")

### Tu turno 0A · La nota de Beto
Beto sacó S1 = 60, S2 = 55, S3 = 65 y Final = 70. Usa la función `nota_final` → guarda el resultado en `nota_beto`.

In [ ]:
nota_beto = None    # TODO: nota_final([...], pesos)
print("Nota de Beto:", nota_beto)

In [ ]:
revisar_0a(nota_beto)

### 0.4 · NumPy: cuentas con vectores y tablas
**NumPy** es la librería de Python para trabajar con números en bloque (piensa en una hoja de cálculo muy rápida). Su pieza clave es el **array**:

- `x * w` multiplica **elemento por elemento**, sin escribir un `for`.
- `x @ w` es el **producto punto**: multiplica parejas y suma. Es exactamente "cada nota por su peso, todo sumado".
- `.shape` dice la forma: `(4,)` es un vector de 4 números; `(3, 4)` es una tabla de 3 filas y 4 columnas.

In [ ]:
import numpy as np                    # already imported in Configuración; repeated so you see it

x = np.array([80, 70, 90, 60])        # Ana's grades as a vector
w = np.array(pesos)                   # the weights as a vector

print("x * w =", x * w)               # element by element
print("suma  =", (x * w).sum())
print("x @ w =", x @ w)               # dot product: the same 76.0 in one step
print("forma de x:", x.shape)

Ahora tres estudiantes a la vez. Cada **fila** es una persona y cada **columna** una nota. `grupo @ w` calcula todas las notas finales con **una sola operación**.

In [ ]:
grupo = np.array([[80, 70, 90, 60],    # Ana
                  [60, 55, 65, 70],    # Beto
                  [95, 85, 75, 90]])   # Carla
print("forma:", grupo.shape, "→ 3 estudiantes × 4 notas")

### Tu turno 0B · Todo el grupo de una vez
Calcula las tres notas finales con `grupo @ w` → `notas_grupo`.

In [ ]:
notas_grupo = None    # TODO: matrix @ vector
print("Notas finales:", notas_grupo)

In [ ]:
revisar_0b(notas_grupo)

### 0.5 · Tu primera gráfica con matplotlib
**matplotlib** dibuja gráficas. El patrón siempre es: crear la figura → dibujar → poner títulos → `plt.show()`.

In [ ]:
aportes = x * w

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))

# Left: how Ana's grade is built
ax[0].bar(partes, aportes, color=AMBAR)
ax[0].set_title(f"Así se arma la nota de Ana: {x @ w:.1f}")
ax[0].set_ylabel("puntos que aporta cada nota")

# Right: three students against an example passing bar of 70
finales = grupo @ w
colores = [MENTA if f >= 70 else CORAL for f in finales]
ax[1].bar(["Ana", "Beto", "Carla"], finales, color=colores)
ax[1].axhline(70, color=TINTA, ls="--", lw=2, label="70 (ejemplo de mínimo para aprobar)")
ax[1].set_ylim(0, 100); ax[1].set_title("¿Quién aprueba?"); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

**Qué deberías ver:** a la izquierda, las cuatro barras que suman 76; a la derecha, Ana (76) y Carla (86) en verde por encima de la línea y Beto (62) en coral por debajo.

Con esto tienes todo el Python que necesitas hoy: **variables, listas, `for`, funciones, arrays de NumPy, `@` y gráficas.**

---
## 1 · La neurona: tu nota final ya es una neurona
📽 Láminas 12–14

**Qué vas a hacer:** ver que una neurona hace exactamente la cuenta de tu nota final, y después usarla para decidir un crédito.

**La idea:** una neurona recibe datos, multiplica cada uno por su **peso**, suma todo, agrega un **sesgo** y decide.

$$z = w_1x_1 + w_2x_2 + w_3x_3 + w_4x_4 + b \qquad\text{o, en corto,}\qquad z = \mathbf{w}\cdot\mathbf{x} + b$$

**Léelo así:**
- $x$ son los **datos** que entran (aquí, tus cuatro notas);
- $w$ son los **pesos**: cuánto importa cada dato (26.67 %, 26.67 %, 26.67 %, 20 %);
- $b$ es el **sesgo**: algo que se suma o resta siempre. Si se aprobara con 70/100 (es solo un ejemplo), $b = -70$: **la exigencia, con el signo cambiado**;
- $z$ es el **puntaje**: positivo → aprueba, negativo → no aprueba. Esa última decisión la toma la **activación** (sección 2).

| Neurona | Tu nota del curso |
|---|---|
| datos $x$ | S1, S2, S3, Final |
| pesos $w$ | 0.2667, 0.2667, 0.2667, 0.20 |
| sesgo $b$ | −70 (ejemplo de mínimo para aprobar) |
| puntaje $z$ | nota − 70 |
| activación (escalón) | $z \ge 0$ → aprueba |

### Ejemplo resuelto: Ana como neurona

In [ ]:
x_ana  = np.array([80, 70, 90, 60])              # inputs: S1, S2, S3, Final
w_nota = np.array([0.8 / 3, 0.8 / 3, 0.8 / 3, 0.2])   # weights: the syllabus percentages
b_nota = -70                                      # bias: the example passing bar, sign flipped

z_ana = w_nota @ x_ana + b_nota                   # the neuron's score
decision = "aprueba" if z_ana >= 0 else "no aprueba"   # step activation

print(f"Nota ponderada  w·x = {w_nota @ x_ana:.1f}")
print(f"Puntaje z = {w_nota @ x_ana:.1f} + ({b_nota}) = {z_ana:+.1f}  →  {decision}")

In [ ]:
# Picture of the same sum: each grade pushes up, the bias pulls down, z is what is left
parts  = list(w_nota * x_ana) + [b_nota]
labels = ["S1", "S2", "S3", "Final", "sesgo b"]
start  = np.concatenate([[0], np.cumsum(parts)[:-1]])
colors = [AMBAR] * 4 + [AZUL]

plt.figure(figsize=(8, 3.8))
plt.bar(labels, parts, bottom=start, color=colors, edgecolor=TINTA)
plt.bar(["z"], [z_ana], color=MENTA if z_ana >= 0 else CORAL, edgecolor=TINTA)
plt.axhline(0, color=TINTA, lw=1)
plt.ylim(0, 82)
plt.title(f"Ana: cada nota suma, el sesgo resta 70 y queda z = {z_ana:+.1f}")
plt.ylabel("puntos"); plt.show()

**Qué deberías ver:** $w\cdot x = 76.0$ y **z = +6.0 → aprueba**. En la gráfica, las barras ámbar suben hasta 76, la azul baja 70 y la verde es lo que queda: 6. (Beto: 62 − 70 = −8 → no aprueba.)

### Tu turno 1A · ¿Cuánto necesita Ana en el entrega final?
Las semanas de Ana aportan 64 puntos. ¿Qué nota en el entrega final la deja **justo** en 70, es decir, con $z = 0$? → `final_necesario`.

<details><summary>Pista</summary>

En el límite $z = 0$: $\;64 + 0.2\cdot F - 70 = 0$. Despeja $F$. También puedes probar valores con `w_nota @ np.array([80, 70, 90, F]) + b_nota`.
</details>

In [ ]:
final_necesario = None    # TODO

In [ ]:
revisar_1a(final_necesario)

### De la nota al crédito: los pesos pueden ser negativos
En tu nota, los pesos los pone el sílabo y todos suman. **En una red de verdad los pesos no los pone nadie: la red los aprende de ejemplos**, y pueden ser negativos (un dato que *resta*).

**Ejemplo resuelto: el comité de crédito.** Una cooperativa evalúa a un socio con tres datos ya escalados (0 = el promedio de los socios):

| Dato | Valor $x$ | Peso $w$ | Qué significa el peso |
|---|---|---|---|
| Ingreso | 0.8 | 1.2 | suma a favor |
| Antigüedad como socio | 0.5 | 0.7 | suma a favor |
| Atrasos previos | −1.0 (menos que el promedio) | −2.0 | los atrasos **restan** |

El sesgo $b = -0.5$ es un comité algo exigente: empieza restando.

In [ ]:
x = np.array([0.8, 0.5, -1.0])    # member A: income, seniority, previous delays (scaled)
w = np.array([1.2, 0.7, -2.0])    # weights: how much the committee trusts each signal
b = -0.5                          # bias: how demanding the committee is

contributions = w * x             # each input times its weight
z = contributions.sum() + b       # weighted sum + bias  (the same as w @ x + b)

for name, c in zip(["Ingreso", "Antigüedad", "Atrasos"], contributions):
    print(f"{name:11s} aporta {c:+.2f}")
print(f"Sesgo       aporta {b:+.2f}")
print(f"Puntaje total z = {z:.2f}  →  {'aprueba' if z >= 0 else 'rechaza'}")

plt.figure(figsize=(7, 3.2))
vals = list(contributions) + [b]
plt.barh(["Ingreso", "Antigüedad", "Atrasos", "Sesgo"], vals, color=[AMBAR if v > 0 else AZUL for v in vals])
plt.axvline(0, color=TINTA, lw=1); plt.gca().invert_yaxis()
plt.title(f"Socio A: z = {z:.2f} (ámbar suma, azul resta)"); plt.xlabel("aporte al puntaje"); plt.show()

**Qué deberías ver:** aportes +0.96, +0.35 y **+2.00**; sesgo −0.50; **z = 2.81 → aprueba**.
Fíjate en los atrasos: un peso negativo (−2.0) por un dato negativo (−1.0) da un aporte **positivo**: tener menos atrasos que el promedio ayuda mucho.

### Tarea 1 · Programa tu neurona
1. Completa la función `neurona(x, w, b)` para que devuelva $z = \mathbf{w}\cdot\mathbf{x} + b$ con **cualquier** vector.
2. Úsala con el **socio B**: ingreso 0.2, antigüedad −0.5, atrasos 1.5 (más atrasos que el promedio), con los mismos `w` y `b`. Guarda el resultado en `z_b`.

<details><summary>Pista</summary>

En NumPy, `w @ x` multiplica parejas y suma (producto punto), como en la sección 0.4. También sirve `(w * x).sum()`.
</details>

In [ ]:
def neurona(x, w, b):
    return None          # TODO: weighted sum plus bias

x_b = np.array([0.2, -0.5, 1.5])
z_b = None               # TODO: use your function with x_b, w and b
print("z del socio B:", z_b)

In [ ]:
revisar_t1(neurona, z_b)

---
## 2 · La activación: de puntaje a decisión
📽 Láminas 16–18

**Qué vas a hacer:** convertir el puntaje $z$ en la salida de la neurona con tres funciones distintas y compararlas.

**La idea:** $z$ todavía no es una decisión. La **función de activación** lo convierte en la salida:

- **Escalón** — un interruptor: todo o nada. $z \ge 0$ → 1 (aprueba); si no → 0.
- **Sigmoide** — un regulador de luz: un valor suave entre 0 y 1 que se lee como **probabilidad**.
- **ReLU** — una válvula: si $z$ es negativo no pasa nada (0); si es positivo pasa igual. Es la más usada dentro de las redes.

$$\sigma(z) = \frac{1}{1+e^{-z}} \qquad \text{ReLU}(z) = \max(0, z)$$

**Léelo así (sigmoide):** "uno dividido entre uno más $e$ a la menos $z$". Si $z$ es muy grande, $e^{-z}$ casi vale 0 y queda $1/1 = 1$; si $z$ es muy negativo, $e^{-z}$ es enorme y queda casi 0; si $z = 0$, queda $1/2$: la neurona duda.

### Ejemplo resuelto: el socio A ($z = 2.81$)

In [ ]:
step    = lambda z: np.where(z >= 0, 1.0, 0.0)   # switch
sigmoid = lambda z: 1 / (1 + np.exp(-z))         # dimmer: 0..1
relu    = lambda z: np.maximum(0, z)             # valve

print(f"Para z = {z:.2f}:  escalón = {step(z):.0f} | sigmoide = {sigmoid(z):.3f} | ReLU = {relu(z):.2f}")
print(f"Para z = 0:     sigmoide = {sigmoid(0):.3f}  (la neurona duda)")
print(f"Para z = -{z:.2f}: sigmoide = {sigmoid(-z):.3f}")

In [ ]:
zz = np.linspace(-6, 6, 300)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, f, name, sub in zip(ax, [step, sigmoid, relu], ["Escalón", "Sigmoide", "ReLU"],
                           ["un interruptor", "un regulador de luz", "una válvula"]):
    a.plot(zz, f(zz), lw=3, color=AMBAR)
    a.axvline(z, ls="--", color="gray"); a.scatter([z], [f(z)], s=60, color=TINTA, zorder=3)
    a.set_title(f"{name}: {sub}"); a.set_xlabel("puntaje z")
ax[0].set_ylabel("salida de la neurona")
plt.suptitle("El mismo z = 2.81 (punto negro) en tres activaciones", y=1.03); plt.tight_layout(); plt.show()

**Qué deberías ver:** escalón = 1 (aprueba), sigmoide ≈ **0.943** (94 % de probabilidad de pago), ReLU = 2.81; con $z = 0$ la sigmoide da 0.5 y con $z = -2.81$, 0.057.

<details><summary>Para curiosos: ¿por qué hace falta la activación?</summary>

Sin activación, apilar capas no sirve: una suma de sumas sigue siendo una suma, y su frontera sigue siendo una línea recta. La activación "dobla" la línea; con muchos dobleces la red arma cualquier curva (lámina 18).
</details>

### Tarea 2 · De puntaje a decisión
Con el `z_b` de la Tarea 1:
1. Calcula la probabilidad de pago del socio B con la sigmoide → `p_b`.
2. Decide con umbral 0.5: si `p_b >= 0.5` escribe `"aprueba"`; si no, `"rechaza"` → `decision_b`.

In [ ]:
p_b = None           # TODO: sigmoid of z_b
decision_b = None    # TODO: "aprueba" or "rechaza"
print("Probabilidad de pago del socio B:", p_b, "→", decision_b)

In [ ]:
revisar_t2(p_b, decision_b)

---
## 3 · El perceptrón corrige UN error
📽 Láminas 21–22

**Qué vas a hacer:** aplicar a mano la regla con la que aprende el perceptrón (Rosenblatt, 1958).

**La idea:** como **sazonar un locro**: pruebas, notas que le falta sal y echas **una pizca**, no medio salero. El perceptrón se corrige **solo cuando se equivoca**, un poquito, en la dirección de la respuesta correcta. Usamos etiquetas $y = +1$ (debía aprobar) y $y = -1$ (debía rechazar):

$$\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot y \cdot \mathbf{x} \qquad\qquad b \leftarrow b + \eta \cdot y$$

**Léelo así:** "el peso nuevo es el peso viejo **más** η (el tamaño de la pizca) **por** $y$ (+1 si debía aprobar, −1 si debía rechazar) **por** $x$ (el dato que clasificó mal)". Si debía aprobar, le *suma* el dato; si debía rechazar, se lo *resta*. Si acertó, no toca nada.

### Ejemplo resuelto: debía aprobar y rechazó ($y = +1$)

In [ ]:
w = np.array([0.2, -0.5]); b = 0.0; eta = 1.0     # current weights
x_i = np.array([1.0, 1.0]); y_i = +1              # a member who SHOULD be approved

z_before = w @ x_i + b
print(f"Antes:   z = {z_before:+.2f} -> predice {'aprobar' if z_before >= 0 else 'rechazar'} (correcto: aprobar)")
w_before, b_before = w.copy(), b

if y_i * z_before <= 0:          # mistake (signs disagree) -> correct the weights
    w = w + eta * y_i * x_i
    b = b + eta * y_i
print(f"Pesos corregidos: w = {w}, b = {b}")

z_after = w @ x_i + b
print(f"Después: z = {z_after:+.2f} -> predice {'aprobar' if z_after >= 0 else 'rechazar'}")

In [ ]:
# The decision line z = 0 before and after the correction
xs = np.linspace(-2, 2, 50)
plt.figure(figsize=(5.6, 5))
plt.plot(xs, -(w_before[0] * xs + b_before) / w_before[1], color=CORAL, lw=2.5, ls="--", label="línea antes (se equivoca)")
plt.plot(xs, -(w[0] * xs + b) / w[1], color=MENTA, lw=3, label="línea después (acierta)")
plt.scatter(*x_i, s=220, color=AMBAR, edgecolor=TINTA, zorder=3, label="el socio (debía aprobar)")
plt.xlim(-2, 2); plt.ylim(-2, 2); plt.xlabel("dato 1"); plt.ylabel("dato 2")
plt.title("Una corrección gira y mueve la línea"); plt.legend(loc="lower left", fontsize=9); plt.show()

**Qué deberías ver:** antes z = **−0.30** (se equivoca y rechaza); después de corregir w = (1.2, 0.5), b = 1 y z = **+2.70** (ahora aprueba). En la gráfica la línea saltó para dejar al socio del lado de "aprobar".

### Tarea 3 · Corrige un error al revés ($y = -1$)
Ahora el comité tiene `w = (1.2, 0.5)` y `b = 1`. Llega un socio `x = (1.0, 0.5)` que **no pagó** ($y = -1$), pero la neurona lo aprueba (z = 2.45).
1. Aplica **una** corrección con η = 1 → `w_nuevo`, `b_nuevo`.
2. Con los pesos nuevos, ¿todavía se equivoca? Escribe `True` o `False` en `todavia_se_equivoca`.

<details><summary>Pista</summary>

Como $y = -1$, la regla **resta** el dato: $\mathbf{w} + 1\cdot(-1)\cdot\mathbf{x} = \mathbf{w} - \mathbf{x}$. Luego calcula `w_nuevo @ x_i + b_nuevo`: si sigue siendo ≥ 0, todavía lo aprueba.
</details>

In [ ]:
w = np.array([1.2, 0.5]); b = 1.0; eta = 1.0
x_i = np.array([1.0, 0.5]); y_i = -1

w_nuevo = None                # TODO: w + eta * y_i * x_i
b_nuevo = None                # TODO: b + eta * y_i
todavia_se_equivoca = None    # TODO: True or False (compute z with the new weights)

In [ ]:
revisar_t3(w_nuevo, b_nuevo, todavia_se_equivoca)

---
## 4 · Entrenar un perceptrón completo
📽 Lámina 23

**Qué vas a hacer:** repetir la regla sobre muchos socios hasta que la neurona deje de equivocarse, y **ver moverse la línea**.

**La idea:** recorrer todos los ejemplos corrigiendo cada error. Cada vuelta completa por todos los datos se llama **época**. Si existe una línea que separa a los que pagan de los que no, el perceptrón la encuentra (teorema de Novikoff, 1962).

### Ejemplo resuelto
Primero generamos 120 socios ficticios con dos datos y una regla "verdadera" que deja un hueco entre los dos grupos.

In [ ]:
np.random.seed(2026)                               # same data for everyone
n = 120
X = np.random.uniform(-1, 1, size=(n, 2))          # 120 members, 2 scaled features
margin = X @ np.array([1.0, 1.6]) - 0.1
keep = np.abs(margin) > 0.1                        # leave a small gap between the classes
X, y = X[keep], np.where(margin[keep] > 0, 1, -1)  # +1 = paid, -1 = did not pay
print("Socios:", X.shape[0], "| pagaron:", (y == 1).sum(), "| no pagaron:", (y == -1).sum())

La función de entrenamiento: dos ciclos `for`, uno por las épocas y otro por los socios. La lista `trace` (opcional) guarda los pesos después de cada corrección para poder dibujar cómo se movió la línea.

In [ ]:
def train_perceptron(X, y, eta=1.0, epochs=50, trace=None):
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            if yi * (w @ xi + b) <= 0:             # wrong (or undecided) -> correct
                w, b = w + eta * yi * xi, b + eta * yi
                mistakes += 1
                if trace is not None:
                    trace.append((w.copy(), b))    # remember the line after this correction
        history.append(mistakes)
        if mistakes == 0:                          # a full pass without errors: done
            break
    return w, b, history

lineas = []
w_p, b_p, hist = train_perceptron(X, y, trace=lineas)
print("Errores por época:", hist, "| correcciones:", len(lineas))

In [ ]:
# Four snapshots of the learning: shaded = what the neuron approves (amber) or rejects (azure)
g1, g2 = np.meshgrid(np.linspace(-1.1, 1.1, 200), np.linspace(-1.1, 1.1, 200))
picks = [0, 2, 4, len(lineas) - 1]                      # after correction 1, 3, 5 and the last one
fig, ax = plt.subplots(1, 4, figsize=(16, 4.3), sharey=True)
for a, k in zip(ax, picks):
    wk, bk = lineas[k]
    zone = (wk[0] * g1 + wk[1] * g2 + bk) >= 0
    a.contourf(g1, g2, zone, levels=[-.5, .5, 1.5], colors=["#D6ECFA", "#FCEBCB"])
    wrong = np.where(X @ wk + bk >= 0, 1, -1) != y
    a.scatter(*X[y == 1].T, c=AMBAR, s=22, edgecolor=TINTA, lw=.3)
    a.scatter(*X[y == -1].T, c=AZUL, s=22, edgecolor=TINTA, lw=.3)
    a.scatter(*X[wrong].T, s=90, facecolors="none", edgecolors=CORAL, linewidths=2)
    a.set_title(f"Tras la corrección {k + 1}: {wrong.sum()} mal clasificados")
    a.set_xlim(-1.1, 1.1); a.set_ylim(-1.1, 1.1); a.set_xlabel("dato 1")
ax[0].set_ylabel("dato 2")
plt.suptitle("Fondo ámbar = la neurona aprueba · fondo azul = rechaza · círculo coral = se equivoca", y=1.04)
plt.tight_layout(); plt.show()

**Qué deberías ver:** errores por época **[8, 0]**: en la primera vuelta se equivoca 8 veces y se corrige cada vez; en la segunda vuelta ya no falla con nadie. En las cuatro fotos, la frontera entre el fondo ámbar y el azul gira y se mueve hasta que ya no queda ningún círculo coral.

### Tarea 4 · Aprende la compuerta AND
AND es "los dos a la vez": vale $+1$ solo si ambos interruptores están en 1.
1. Escribe los 4 casos en `X_and` y sus respuestas en `y_and` (+1 o −1).
2. Entrena con la función `train_perceptron` de arriba → `w_and, b_and, hist_and`.

| $x_1$ | $x_2$ | AND |
|---|---|---|
| 0 | 0 | −1 |
| 0 | 1 | −1 |
| 1 | 0 | −1 |
| 1 | 1 | +1 |

In [ ]:
X_and = None    # TODO: np.array with the 4 rows [x1, x2]
y_and = None    # TODO: np.array with -1 / +1
w_and, b_and, hist_and = None, None, None    # TODO: train_perceptron(X_and, y_and)

In [ ]:
revisar_t4(w_and, b_and, X_and, y_and)

---
## 5 · El límite: XOR, la luz de la escalera
📽 Lámina 24

**Qué vas a hacer:** pedirle al perceptrón algo imposible y ver por qué falla.

**La idea:** dos interruptores, uno abajo y otro arriba de la escalera. La luz se enciende si están en posiciones **distintas**. Eso es XOR.

| $x_1$ | $x_2$ | XOR |
|---|---|---|
| 0 | 0 | apagada (−1) |
| 0 | 1 | **encendida (+1)** |
| 1 | 0 | **encendida (+1)** |
| 1 | 1 | apagada (−1) |

### Ejemplo resuelto: el perceptrón intenta aprender XOR

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_xor = np.array([-1, 1, 1, -1])
w_x, b_x, hist_xor = train_perceptron(X_xor, y_xor, epochs=20)
print("Errores por época:", hist_xor)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(range(1, len(hist_xor) + 1), hist_xor, "o-", color=CORAL, lw=2)
ax[0].set_ylim(0, 4.3); ax[0].set_xlabel("época"); ax[0].set_ylabel("errores")
ax[0].set_title("XOR: nunca llega a 0 errores")

# Even the best possible line (x1 + x2 = 0.5) gets one case wrong
w_best, b_best = np.array([1.0, 1.0]), -0.5
g1, g2 = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
ax[1].contourf(g1, g2, (w_best[0] * g1 + w_best[1] * g2 + b_best) >= 0, levels=[-.5, .5, 1.5], colors=["#D6ECFA", "#FCEBCB"])
pred = np.where(X_xor @ w_best + b_best >= 0, 1, -1)
for (a, b_), t, p in zip(X_xor, y_xor, pred):
    ax[1].scatter(a, b_, s=320, color=AMBAR if t == 1 else AZUL, edgecolor=CORAL if p != t else TINTA,
                  linewidth=4 if p != t else 1, zorder=3)
xs = np.linspace(-0.5, 1.5, 50)
ax[1].plot(xs, 0.5 - xs, color=TINTA, lw=2.5)
ax[1].set_xlim(-0.5, 1.5); ax[1].set_ylim(-0.5, 1.5); ax[1].set_xlabel("interruptor 1"); ax[1].set_ylabel("interruptor 2")
ax[1].set_title("La mejor línea posible acierta 3 de 4 (borde coral = error)")
plt.tight_layout(); plt.show()

**Qué deberías ver:** **4 errores en todas las épocas**: nunca mejora. A la derecha, incluso la mejor línea posible deja un caso del lado equivocado (borde coral): las dos luces encendidas (ámbar) están en esquinas opuestas y ninguna recta las junta sin llevarse una apagada.

### Tarea 5 · ¿Por qué falla?
Elige la mejor explicación y escribe su letra en `respuesta_t5`:

- **a)** Le faltan épocas: con 1 000 épocas lo lograría.
- **b)** No existe una línea recta que separe las luces encendidas de las apagadas.
- **c)** La tasa de aprendizaje η es demasiado grande.
- **d)** Los datos tienen ruido.

En el negocio esto aparece cuando el efecto de una variable **depende de otra** (por ejemplo, el garante ayuda solo si la cuota es baja). Mañana lo resolvemos con **más de una neurona**.

In [ ]:
respuesta_t5 = None    # TODO: "a", "b", "c" or "d"

In [ ]:
revisar_t5(respuesta_t5)

---
## 6 · La pérdida: ¿qué tan mal lo hicimos?
📽 Láminas 28–29

**Qué vas a hacer:** medir el error con **un solo número**, para predecir precios (MSE) y para predecir sí/no (BCE).

**La idea:** la pérdida es como una nota al revés: **0 es perfecto**, más alto es peor.

**Error cuadrático medio (MSE)**, para predecir números:
$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N} (y_i - \hat y_i)^2$$
**Léelo así:** "para cada caso, resta lo real ($y$) menos lo predicho ($\hat y$, se lee *y gorro*), eleva al cuadrado y saca el promedio". Al cuadrado, los negativos no restan y los errores grandes pesan muchísimo.

**Entropía cruzada binaria (BCE)**, para predecir sí/no con una probabilidad $\hat y$:
$$\text{BCE} = -\big[\,y\log\hat y + (1-y)\log(1-\hat y)\,\big]$$
**Léelo así:** "menos el logaritmo de la probabilidad que el modelo le dio a lo que de verdad pasó". Si pagó ($y=1$) queda $-\log\hat y$; si no pagó ($y=0$) queda $-\log(1-\hat y)$. Como el **meteorólogo** que dijo "99 % sin lluvia" el día del aguacero: estar **seguro y equivocado** cuesta carísimo.

### Ejemplo resuelto

In [ ]:
# MSE: predicted vs real house prices in Quito (thousands of USD)
real = np.array([120, 95, 150, 80])
pred = np.array([110, 100, 170, 82])
errors = real - pred
print("Errores:", errors, "| al cuadrado:", errors**2, "| MSE =", np.mean(errors**2))

# BCE: the real answer is y = 1 ("sí pagó"). How much does each prediction cost?
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"El modelo dijo {p:4.2f} -> pérdida = -log({p}) = {-np.log(p):.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
casas = ["Casa 1", "Casa 2", "Casa 3", "Casa 4"]
pos = np.arange(4)
ax[0].bar(pos - .2, np.abs(errors), .4, color=AZUL, label="|error| (miles USD)")
ax[0].bar(pos + .2, errors**2, .4, color=AMBAR, label="error al cuadrado")
ax[0].axhline(np.mean(errors**2), color=TINTA, ls="--", label=f"MSE = {np.mean(errors**2):.2f}")
ax[0].set_xticks(pos, casas); ax[0].legend(); ax[0].set_title("MSE: la casa 3 pesa casi todo")

ps = np.linspace(0.005, 1, 300)
ax[1].plot(ps, -np.log(ps), color=CORAL, lw=3)
for p in [0.99, 0.5, 0.01]:
    ax[1].scatter(p, -np.log(p), s=70, color=TINTA, zorder=3)
    ax[1].annotate(f"dijo {p:.0%} → {-np.log(p):.2f}", (p, -np.log(p)), textcoords="offset points", xytext=(10, 6))
ax[1].set_xlabel("probabilidad que le dio a «sí pagó» (y sí pagó)"); ax[1].set_ylabel("pérdida = −log(p)")
ax[1].set_title("BCE: estar seguro y equivocado se dispara")
plt.tight_layout(); plt.show()

**Qué deberías ver:** MSE = **132.25** (la casa 3, con error 20, aporta 400 de los 529). En la BCE: 0.99 → **0.01**, 0.5 → **0.69** y 0.01 → **4.61**; la curva coral sube en picada cuando el modelo estaba seguro de lo contrario.

### Tarea 6 · Programa MSE y BCE
1. `mse(real, pred)`: el promedio de los errores al cuadrado.
2. `bce(y, p)`: la fórmula completa de arriba, para **un** dato con $y \in \{0, 1\}$.

<details><summary>Pista</summary>

`np.mean(...)`, `** 2` y `np.log(...)` son todo lo que necesitas.
</details>

In [ ]:
def mse(real, pred):
    return None    # TODO

def bce(y, p):
    return None    # TODO

In [ ]:
revisar_t6(mse, bce)

---
## 7 · Descenso de gradiente: bajar el Cotopaxi con neblina
📽 Láminas 30–32

**Qué vas a hacer:** bajar por una curva de pérdida dando pasos, como lo hace cualquier red al entrenar.

**La idea:** estás en la ladera del Cotopaxi con neblina y quieres llegar a la laguna de abajo. No ves el camino, pero con el pie sientes hacia dónde baja el terreno (la **pendiente**) y das un paso hacia allá. Vuelves a tantear y das otro.

| En la montaña | En el entrenamiento |
|---|---|
| la altura | la pérdida |
| dónde estás parado | los pesos $w$ |
| la inclinación bajo tus pies | la pendiente (gradiente) |
| el largo de tu paso | η, la tasa de aprendizaje |

$$w_{\text{nuevo}} = w - \eta \cdot \text{pendiente}$$

**Léelo así:** "el peso nuevo es el peso actual **menos** η por la pendiente". Se **resta** porque la pendiente apunta cuesta **arriba**: si la pendiente es negativa, $w$ sube; si es positiva, $w$ baja.

### Ejemplo resuelto
Pérdida $L(w) = (w - 3)^2$: su fondo está en $w = 3$. Su pendiente (derivada) es $2(w-3)$. Empezamos en $w = 0$ con $\eta = 0.1$.

In [ ]:
L = lambda w: (w - 3) ** 2          # loss: a valley with its bottom at w = 3
slope = lambda w: 2 * (w - 3)       # its slope (derivative)

w, eta, path = 0.0, 0.1, [0.0]
print(" paso |    w    | pendiente | pérdida")
for step_i in range(10):
    print(f"  {step_i:2d}  | {w:7.3f} | {slope(w):9.3f} | {L(w):7.3f}")
    w = w - eta * slope(w)           # one step downhill
    path.append(w)

In [ ]:
ws = np.linspace(-0.5, 5, 200)
plt.figure(figsize=(7.5, 4))
plt.plot(ws, L(ws), lw=3, color=CORAL, label="pérdida L(w) = (w − 3)²")
plt.plot(path, [L(p) for p in path], "o-", color=AMBAR, ms=7, lw=2, label="los pasos (η = 0.1)")
plt.annotate("inicio", (path[0], L(path[0])), textcoords="offset points", xytext=(10, 0))
plt.scatter([3], [0], s=120, marker="*", color=MENTA, zorder=3, label="el fondo: w = 3")
plt.xlabel("w (dónde estás)"); plt.ylabel("pérdida (altura)"); plt.legend()
plt.title("Cada paso baja un poco, y los pasos se acortan solos"); plt.show()

**Qué deberías ver:** w = 0 → 0.6 → 1.08 → 1.464 → 1.771 … acercándose a **3**, y la pérdida bajando de 9 hacia 0. Los pasos se acortan solos: cerca del fondo la pendiente es menor.

### Tarea 7 · Baja otra montaña
Ahora la pérdida es $L(w) = (w + 2)^2$.
1. ¿En qué valor de $w$ está el fondo? → `minimo`.
2. Su pendiente es $2(w+2)$. Empieza en $w = 5$ y da **50 pasos** con $\eta = 0.1$. Guarda el último valor en `w_final`.

<details><summary>Pista</summary>

Copia el ciclo del ejemplo y cambia la pendiente: `w = w - eta * 2 * (w + 2)`.
</details>

In [ ]:
minimo = None     # TODO: where is the bottom?
w = 5.0
eta = 0.1
# TODO: write a loop of 50 steps: w = w - eta * slope, with slope = 2 * (w + 2)

w_final = None    # TODO: the value of w after the loop

In [ ]:
revisar_t7(minimo, w_final)

---
## 8 · El tamaño del paso η importa
📽 Lámina 33

**Qué vas a hacer:** bajar la misma montaña con tres tamaños de paso y comparar.

**La idea:** como **la ducha con agua caliente**: si giras la llave poquito a poco, tardas pero llegas a la temperatura justa; si la abres de golpe te quemas, la cierras de golpe te congelas… y nunca llegas. Con η muy pequeño tardas una eternidad; con η muy grande te pasas del fondo y rebotas cada vez más lejos.

### Ejemplo resuelto: misma montaña, tres pasos distintos

In [ ]:
def descenso(eta, pasos=30, w=0.0):
    # gradient descent on L(w) = (w - 3)^2; returns the final w
    for _ in range(pasos):
        w = w - eta * slope(w)
    return w

for eta in [0.02, 0.4, 1.02]:
    print(f"η = {eta:4}: después de 30 pasos w = {descenso(eta):10.3f}  (el fondo está en 3)")

In [ ]:
ws_wide = np.linspace(-3, 9, 300)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for a, eta, title in zip(ax, [0.02, 0.4, 1.02], ["muy pequeño: lento", "bueno: llega", "muy grande: diverge"]):
    w, path = 0.0, [0.0]
    for _ in range(12):
        w = w - eta * slope(w); path.append(w)
    a.plot(ws_wide, L(ws_wide), lw=2.5, color=CORAL)
    a.plot(path, [L(p) for p in path], "o-", color=AMBAR, ms=5)
    a.set_ylim(-1, 40); a.set_title(f"η = {eta}: {title}"); a.set_xlabel("w")
ax[0].set_ylabel("pérdida")
plt.suptitle("12 pasos con cada η", y=1.03); plt.tight_layout(); plt.show()

**Qué deberías ver:** con η = 0.02 apenas avanza; con η = 0.4 llega rápido; con η = 1.02 **rebota de lado a lado, cada vez más lejos**: la pérdida crece en vez de bajar (diverge). En Keras este número se llama `learning_rate`: si la pérdida sube o aparece `nan`, es lo primero que se revisa.

### Tarea 8 · ¿Cuáles convergen?
Usa `descenso(eta)` para probar cada valor y escribe `True` si **se acerca** al fondo (aunque sea lento o rebotando) o `False` si **se aleja**.

In [ ]:
# try: print(descenso(0.9))
converge = {0.05: None, 0.5: None, 0.9: None, 1.1: None}    # TODO: True / False

In [ ]:
revisar_t8(converge)

---
## Resumen en 5 líneas
1. Una neurona **multiplica, suma y decide**: $z = \mathbf{w}\cdot\mathbf{x} + b$, como tu nota final (pesos = porcentajes, sesgo = la exigencia).
2. La **activación** convierte el puntaje en la salida: escalón (sí/no), sigmoide (probabilidad) o ReLU.
3. El perceptrón aprende **corrigiendo sus errores** con una pizca (η), pero una sola neurona solo traza **una línea**: no puede con XOR.
4. La **pérdida** mide el error con un número: MSE para números, BCE para sí/no.
5. El **descenso de gradiente** baja la pérdida paso a paso en contra de la pendiente; el tamaño del paso η decide si llegas, tardas o divergen.

**Mañana:** juntamos neuronas en capas, vemos cómo aprende una red completa (retropropagación) y pasamos a Keras con el **NB02**.

---
## 🏁 Tareas finales
Tres tareas para practicar en clase o en casa. Cada una se revisa sola.

### Tarea final 1 · La nota de todo el grupo
`GRUPO6` (ya cargado) tiene las notas de 6 estudiantes (S1, S2, S3, Final), una fila por estudiante.
1. Calcula sus 6 notas finales con la neurona de la sección 1 → `notas6` (usa `@` con `w_nota`).
2. ¿Cuántos aprueban con el mínimo de ejemplo de 70, es decir, con $z = \text{nota} - 70 \ge 0$? → `aprobados`.

In [ ]:
print(GRUPO6)
notas6 = None       # TODO: GRUPO6 @ w_nota
aprobados = None    # TODO: how many have z >= 0

In [ ]:
revisar_tf2(notas6, aprobados)

### Tarea final 2 · Elige tu paso
Con la función `descenso(eta, pasos)` de la sección 8, busca un η que llegue a **menos de 0.01 del fondo (w = 3) en solo 10 pasos** → `mi_eta`. Prueba varios con `print(descenso(0.3, pasos=10))`.

In [ ]:
mi_eta = None    # TODO: try a few values first

In [ ]:
revisar_tf3(descenso, mi_eta)

### Tarea final 3 · Mini-reto 1 (opcional): tu propio comité de crédito
Seis socios de la cooperativa, con dos datos escalados: **ingreso** y **atrasos previos**. Tres pagaron (ámbar) y tres no (azul).

| Socio | Ingreso | Atrasos | ¿Pagó? |
|---|---|---|---|
| Ana | 0.9 | −0.8 | sí |
| Byron | 0.4 | −0.2 | sí |
| Carla | 0.8 | 0.3 | sí |
| Diego | −0.5 | 0.6 | no |
| Elena | 0.1 | 0.5 | no |
| Fabián | −0.3 | −0.1 | no |

**Tu misión:** elige a mano los pesos `w = [w_ingreso, w_atrasos]` y el sesgo `b` para que tu neurona apruebe a los que pagaron y rechace a los que no. La celda dibuja tu frontera y cuenta aciertos.

**Para pensar:**
1. ¿Qué signo debería tener el peso de los atrasos? ¿Por qué?
2. Carla está cerca de la frontera: ¿qué le pasa si haces al comité más exigente (bajas $b$)?
3. **Bonus:** entrena con `train_perceptron(SOCIOS, PAGO)` y compara sus pesos con los tuyos.

In [ ]:
w_comite = None    # TODO: e.g. [1.0, -1.0]
b_comite = None    # TODO: e.g. 0.0
revisar_mini_reto_1(w_comite, b_comite)

---
**¿Te quedó alguna duda?** Repasa las slides interactivas y la guía de estudio desde la página del curso. Mañana seguimos con el **NB02 · Tu primera red**.